# Week 4: In-Context Learning and Training Data Generation

In this practical, we will explore:

1. **Chat Templates**: Understanding how LLMs format conversations
   - Exploring `apply_chat_template` and its parameters
   - Introduction to thinking models (DeepSeek-R1, Qwen3)
1. **In-Context Learning (ICL)**: Learning from examples in the prompt
   - Chain-of-Thought (CoT) prompting for step-by-step reasoning
   - Self-Consistency for improved reliability
1. **Dataset Exploration**: Working with the LoTTE IR benchmark
1. **Training Data Generation**: Creating synthetic queries for IR training
   - Validating generated queries using ROUGE and BERTScore
1. **Introduction to RAG**: Simple RAG with BM25 retrieval

We use the LoTTE (Long-Tail Topic-stratified Evaluation) dataset,
which contains StackExchange Q&A pairs with search queries.

## Setup

In [1]:
import shutil
import torch
import pyterrier as pt
import pandas as pd
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from evaluate import load as load_metric


d:\Sorbonne\M2-MIND\LLM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_best_device():
    """Returns the best device on this computer"""

    if torch.cuda.is_available():
        device = torch.device("cuda")
        total_memory = torch.cuda.get_device_properties(device).total_memory
        print(f"GPU Memory: {total_memory / 1e9:.1f} GB")
        print(f"GPU Name: {torch.cuda.get_device_name(device)}")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    print(f"Found device: {device}")
    return device


device = get_best_device()

GPU Memory: 8.6 GB
GPU Name: NVIDIA GeForce RTX 3070
Found device: cuda


In [4]:
# Load a model for generation
#
# Options (uncomment ONE model_name):
#
# 1. SmolLM2-1.7B float16 (~3.4GB) - default, works on all platforms
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
#
# 2. Pre-quantized models (GPTQ/AWQ) - Linux only, requires:
#    uv pip install auto-gptq autoawq  (or: uv sync --extra quantized)
# FIXME: update to Qwen3 and test
# model_name = "Qwen/Qwen2.5-3B-Instruct-AWQ"  # 3B AWQ, ~2GB
# model_name = "Qwen/Qwen2.5-7B-Instruct-AWQ"  # 7B AWQ, ~4GB

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Detect if model is pre-quantized (AWQ/GPTQ) by name
is_quantized = "AWQ" in model_name or "GPTQ" in model_name

if is_quantized:
    # Pre-quantized models need autoawq/auto-gptq (Linux only)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
    )
    print(f"Model loaded: {model_name} (pre-quantized)")
else:
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float16,
    )
    model = model.to(device)
    print(f"Model loaded: {model_name} on {device}")

model.eval()

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Model loaded: HuggingFaceTB/SmolLM2-1.7B-Instruct on cuda


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), ep

In [5]:
def build_prompt(user: str, system: str = "You are a helpful assistant.") -> str:
    """Build a chat-format prompt using the tokenizer's chat template."""
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


@torch.no_grad()
def generate_text(
    prompt: str,
    max_new_tokens: int = 100,
    temperature: float = 0.7,
    do_sample: bool = True,
    num_return_sequences: int = 1,
) -> list[str]:
    """Generate text from a prompt."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
        num_return_sequences=num_return_sequences,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode and extract only the response
    generated = []
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        # Extract the part after the last "assistant"
        if "assistant" in text.lower():
            text = text.split("assistant")[-1].strip()
        generated.append(text)

    return generated

## Part 0: Understanding Chat Templates

Modern instruction-tuned LLMs are trained with specific **chat formats**
that structure conversations between users and assistants. The
`apply_chat_template` method from HuggingFace tokenizers handles this
formatting automatically.

### Why chat templates matter:
- Different models use different formats (ChatML, Llama, etc.)
- Incorrect formatting leads to poor model performance
- Templates handle special tokens automatically

### Components of a chat:
- **system**: Instructions that define the assistant's behavior
- **user**: Messages from the user
- **assistant**: Responses from the model

### 0.1 Examining the Chat Template

Let's see what the chat template for our model looks like.

In [6]:
# The chat template is a Jinja2 template stored in the tokenizer
print(f"Chat template for {model_name}:")
print("-" * 60)
print(tokenizer.chat_template)

Chat template for HuggingFaceTB/SmolLM2-1.7B-Instruct:
------------------------------------------------------------
{% for message in messages %}{% if loop.first and messages[0]['role'] != 'system' %}{{ '<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
' }}{% endif %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
' }}{% endif %}


In [7]:
# Let's see how a simple conversation gets formatted
simple_messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]

# Without add_generation_prompt
formatted = tokenizer.apply_chat_template(simple_messages, tokenize=False)
print("Formatted (without generation prompt):")
print(repr(formatted))
print()

# With add_generation_prompt - adds the assistant's turn start
formatted_gen = tokenizer.apply_chat_template(
    simple_messages, tokenize=False, add_generation_prompt=True
)
print("Formatted (with generation prompt):")
print(repr(formatted_gen))

Formatted (without generation prompt):
'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n'

Formatted (with generation prompt):
'<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nWhat is 2+2?<|im_end|>\n<|im_start|>assistant\n'


The `add_generation_prompt=True` option adds tokens that signal to the model
it should start generating a response. This is crucial for inference!

In [8]:
# Multi-turn conversation example
multi_turn = [
    {"role": "system", "content": "You are a math tutor."},
    {"role": "user", "content": "What is 5 + 3?"},
    {"role": "assistant", "content": "5 + 3 = 8"},
    {"role": "user", "content": "And what is 8 × 2?"},
]

formatted_multi = tokenizer.apply_chat_template(
    multi_turn, tokenize=False, add_generation_prompt=True
)
print("Multi-turn conversation:")
print(formatted_multi)

Multi-turn conversation:
<|im_start|>system
You are a math tutor.<|im_end|>
<|im_start|>user
What is 5 + 3?<|im_end|>
<|im_start|>assistant
5 + 3 = 8<|im_end|>
<|im_start|>user
And what is 8 × 2?<|im_end|>
<|im_start|>assistant



### 0.2 Thinking Models

Some recent models support explicit **thinking/reasoning tokens**:
- **DeepSeek-R1**: Uses `<think>...</think>` for chain-of-thought reasoning
- **Qwen3**: Supports `enable_thinking=True` in `apply_chat_template`

These models can show their reasoning process explicitly, which is useful
for debugging and understanding model behavior.

Let's load a Qwen3 tokenizer to explore this feature (we only need the
tokenizer, not the full model).

In [9]:
# Load a Qwen3 tokenizer to explore thinking mode
# (Qwen3 is a "thinking" model with enable_thinking support)
thinking_model_name = "Qwen/Qwen3-0.6B"
thinking_tokenizer = AutoTokenizer.from_pretrained(thinking_model_name)

print(f"Loaded tokenizer for: {thinking_model_name}")

d:\Sorbonne\M2-MIND\LLM\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\titou\.cache\huggingface\hub\models--Qwen--Qwen3-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For

Loaded tokenizer for: Qwen/Qwen3-0.6B


In [10]:
# Compare prompts with and without thinking enabled
test_messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

# Without thinking (standard mode)
prompt_no_think = thinking_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
print("=== Without thinking (enable_thinking=False) ===")
print(prompt_no_think)

=== Without thinking (enable_thinking=False) ===
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant
<think>

</think>




In [11]:
# With thinking enabled - the model will generate <think>...</think> blocks
prompt_with_think = thinking_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)
print("=== With thinking (enable_thinking=True) ===")
print(prompt_with_think)

=== With thinking (enable_thinking=True) ===
<|im_start|>user
What is the capital of France?<|im_end|>
<|im_start|>assistant



Notice how the prompt differs when thinking is enabled. The model is
instructed to first reason in `<think>` tags before providing the answer.

**Example of thinking model output:**
```
<think>
The user is asking about the capital of France.
France is a country in Western Europe.
Its capital city is Paris, which is also its largest city.
</think>

The capital of France is Paris.
```

### Exercise 0: Explore different chat formats

Try formatting the same message with different configurations and observe
the differences. What special tokens does each model use?

In [ ]:
# Explore the chat template

messages = [{"role": "user", "content": "Hello!"}]
formatted = tokenizer.apply_chat_template(messages, tokenize=False)
print(formatted)
assert False, 'Not implemented yet'


## Part 1: Introduction to In-Context Learning (ICL)

ICL allows adapting a model to a task simply by providing examples
in the prompt, without modifying the model weights.

### Advantages of ICL:
- No training required
- Flexible: change task by changing examples
- Works with pre-trained models

### Disadvantages:
- Consumes context tokens
- Performance depends on example quality
- Variance in responses

### 1.1 Zero-shot vs Few-shot

Let's start with a sentiment classification task on tech-related text.

In [ ]:
# Example tech support questions (simplified)
test_questions = [
    "How do I fix the blue screen error on Windows?",
    "Python is such a terrible language, nothing works!",
    "Can someone explain how to set up a VPN?",
    "This software is garbage, crashes every 5 minutes.",
]

# Expected labels: question, complaint, question, complaint
expected_labels = ["question", "complaint", "question", "complaint"]

### ZERO-SHOT (no examples)

In [ ]:
# Zero-shot: classification without examples

zero_shot_prompt = """Classify the following text as "question" or "complaint".
Reply with a single word only.

Text: {text}
Classification:"""

for text, expected in zip(test_questions[:2], expected_labels[:2]):
    prompt = build_prompt(zero_shot_prompt.format(text=text))
    response = generate_text(prompt, max_new_tokens=10, temperature=0.1)[0]
    print(f"Text: {text[:60]}...")
    print(f"Predicted: {response.strip()} | Expected: {expected}")
    print()

### FEW-SHOT (with 2 examples)

In [ ]:
# Few-shot: classification with examples

few_shot_examples = """Example 1:
Text: "Where can I find the settings menu in this app?"
Classification: question

Example 2:
Text: "This update broke everything and I lost all my data!"
Classification: complaint

"""

few_shot_prompt = (
    few_shot_examples
    + """Now, classify this text:
Text: "{text}"
Classification:"""
)

for text, expected in zip(test_questions[:2], expected_labels[:2]):
    prompt = build_prompt(few_shot_prompt.format(text=text))
    response = generate_text(prompt, max_new_tokens=10, temperature=0.1)[0]
    print(f"Text: {text[:60]}...")
    print(f"Predicted: {response.strip()} | Expected: {expected}")
    print()

### Exercise 1: Implement a few-shot prompt builder

Complete the following function to automatically build few-shot prompts.

In [ ]:
def build_few_shot_prompt(
    examples: List[Tuple[str, str]],
    query: str,
    task_description: str = "Classify the following text.",
    input_label: str = "Text",
    output_label: str = "Classification",
) -> str:
    """
    Build a few-shot prompt from examples.

    Args:
        examples: List of tuples (input, output) for examples
        query: The text to classify
        task_description: Description of the task
        input_label: Label for input (e.g., "Text", "Question")
        output_label: Label for output (e.g., "Classification", "Answer")

    Returns:
        Formatted prompt for the model
    """
    prompt_parts = [task_description, ""]

    for i, (inp, out) in enumerate(examples, 1):
        prompt_parts.append(f"Example {i}:")
        prompt_parts.append(f'{input_label}: "{inp}"')
        prompt_parts.append(f"{output_label}: {out}")
        prompt_parts.append("")

    prompt_parts.append("Now:")
    prompt_parts.append(f'{input_label}: "{query}"')
    prompt_parts.append(f"{output_label}:")

    return "\n".join(prompt_parts)

In [ ]:
# Test the function
examples = [
    ("How do I update my graphics drivers?", "question"),
    ("The app is so slow it's unusable.", "complaint"),
    ("What's the difference between RAM and ROM?", "question"),
]

test_query = "This product is a complete waste of money."
prompt = build_few_shot_prompt(
    examples,
    test_query,
    task_description="Classify the text as 'question' or 'complaint'.",
)
print("Generated prompt:")
print(prompt)

## Part 1b: Chain-of-Thought (CoT) Prompting

Chain-of-Thought prompting encourages the model to **reason step by step**
before giving a final answer. This improves performance on tasks requiring
multi-step reasoning.

### Key insight:
Instead of asking for a direct answer, we ask the model to "think aloud".
The reasoning process helps the model arrive at better answers.

### Two approaches:
1. **Zero-shot CoT**: Add "Let's think step by step" to the prompt
2. **Few-shot CoT**: Provide examples with explicit reasoning steps

### 1b.1 Zero-shot Chain-of-Thought

Simply adding "Let's think step by step" can improve reasoning.

In [ ]:
# A task that benefits from reasoning
reasoning_questions = [
    # Query 1
    "A store has 45 apples. "
    "They sell 12 in the morning and receive 20 more. "
    "How many apples do they have?",
    # Query 2
    "If a train travels at 60 km/h for 2.5 hours, how far does it travel?",
    # Query 3
    "Marie has 3 times as many books as Paul. "
    "Paul has 8 books. How many books does Marie have?",
]

expected_answers = ["53", "150", "24"]

### DIRECT PROMPTING (no reasoning)

In [ ]:
# Direct prompting (no CoT)

direct_prompt = """Answer the following question with just the number.

Question: {question}
Answer:"""

for question, expected in zip(reasoning_questions[:2], expected_answers[:2]):
    prompt = build_prompt(direct_prompt.format(question=question))
    response = generate_text(prompt, max_new_tokens=20, temperature=0.1)[0]
    print("=" * 80)
    print(f"Q: {question}")
    print(f"A: {response.strip()} (expected: {expected})")
    print()

### ZERO-SHOT CoT ('Let's think step by step')

In [ ]:
# Zero-shot CoT

cot_prompt = """Answer the following question. Let's think step by step.

Question: {question}

Step-by-step reasoning:"""

for question, expected in zip(reasoning_questions[:2], expected_answers[:2]):
    prompt = build_prompt(cot_prompt.format(question=question))
    response = generate_text(prompt, max_new_tokens=150, temperature=0.1)[0]
    print("=" * 80)
    print(f"Q: {question}")
    print(f"Reasoning: {response.strip()}")
    print(f"(Expected answer: {expected})")
    print()

### 1b.2 Few-shot Chain-of-Thought

Providing examples with explicit reasoning steps teaches the model
how to reason about similar problems.

In [ ]:
def build_cot_prompt(
    examples: List[Tuple[str, str, str]],
    query: str,
    task_description: str = "Solve the following problem step by step.",
) -> str:
    """
    Build a few-shot CoT prompt with reasoning examples.

    Args:
        examples: List of (question, reasoning, answer) tuples
        query: The question to solve
        task_description: Description of the task

    Returns:
        Formatted CoT prompt
    """
    # Implement few-shot CoT prompt builder

    # 1. Start with task_description
    # 2. Add each example with Question, Reasoning, and Answer
    # 3. End with the query (without answer)
    prompt_parts = [task_description, ""]
    for i, (question, reasoning, answer) in enumerate(examples, 1):
    assert False, 'Not implemented yet'


### FEW-SHOT CoT (with reasoning examples)

In [ ]:
# Few-shot CoT examples with reasoning

cot_examples = [
    (
        # Query
        "A baker makes 24 cookies. He gives away 8 and bakes 15 more. "
        "How many cookies does he have?",
        # Think
        "Start with 24 cookies. Give away 8: 24 - 8 = 16. Bake 15 more: 16 + 15 = 31.",
        # Answer
        "31",
    ),
    (
        "A car travels at 80 km/h for 3 hours. How far does it go?",
        "Distance = speed × time. Distance = 80 × 3 = 240 km.",
        "240",
    ),
]

for question, expected in zip(reasoning_questions, expected_answers):
    prompt = build_cot_prompt(cot_examples, question)
    full_prompt = build_prompt(prompt)
    response = generate_text(full_prompt, max_new_tokens=100, temperature=0.1)[0]
    print("=" * 80)
    print(f"Q: {question}")
    print(f"Response: {response.strip()}")
    print(f"(Expected: {expected})")
    print()

### Extract final answer from CoT response

CoT responses contain reasoning followed by the answer. We need to extract
the final answer for evaluation.

In [ ]:
def extract_answer_from_cot(response: str) -> str:
    """
    Extract the final numerical answer from a CoT response.

    Args:
        response: The model's response with reasoning

    Returns:
        The extracted answer (number as string)
    """
    import re

    # Try to find "Answer: X" pattern
    answer_match = re.search(r"Answer:\s*(\d+)", response, re.IGNORECASE)
    if answer_match:
        return answer_match.group(1)

    # Try to find "= X" at the end of reasoning
    equals_match = re.search(r"=\s*(\d+)", response)
    if equals_match:
        return equals_match.group(1)

    # Fall back to last number in response
    numbers = re.findall(r"\d+", response)
    if numbers:
        return numbers[-1]

    return ""

In [ ]:
# Test answer extraction
test_responses = [
    "Start with 45. Subtract 12: 45 - 12 = 33. Add 20: 33 + 20 = 53. Answer: 53",
    "Speed is 60 km/h, time is 2.5 hours. Distance = 60 × 2.5 = 150 km.",
    "Paul has 8 books. Marie has 3 times more: 3 × 8 = 24 books.",
]

print("Testing answer extraction:")
for resp in test_responses:
    answer = extract_answer_from_cot(resp)
    print(f"  Response: {resp[:50]}...")
    print(f"  Extracted: {answer}")
    print()

## Part 1c: Self-Consistency Decoding

Self-consistency combines well with CoT: instead of generating one reasoning
path, we generate **multiple paths** and take the **majority answer**.

The idea: different reasoning paths may lead to different answers,
but the correct answer is more likely to appear consistently.

This reduces variance and improves reliability.

In [ ]:
def self_consistency_classify(
    prompt: str,
    num_samples: int = 5,
    temperature: float = 0.7,
    valid_labels: List[str] = None,
) -> Tuple[str, dict]:
    """
    Classification with self-consistency.

    Args:
        prompt: The classification prompt
        num_samples: Number of samples to generate
        temperature: Temperature for sampling
        valid_labels: List of valid labels (for filtering)

    Returns:
        Tuple (majority_label, counts)
    """
    # Implement self-consistency

    responses = generate_text(prompt, ..., num_return_sequences=num_samples)
    labels = []
    for response in responses:
    assert False, 'Not implemented yet'


### SELF-CONSISTENCY (5 samples)

In [ ]:
for text, expected in zip(test_questions, expected_labels):
    prompt = build_prompt(few_shot_prompt.format(text=text))
    label, counts = self_consistency_classify(
        prompt, num_samples=5, valid_labels=["question", "complaint"]
    )
    print(f"Text: {text[:50]}...")
    print(f"Votes: {counts}")
    print(f"Result: {label} | Expected: {expected}")
    print()

### Exercise 2: Compare single-shot vs self-consistency

Measure the accuracy of both approaches on a small test set.

In [ ]:
# Create a test dataset
test_data = [
    ("How do I install Python on Mac?", "question"),
    ("This keyboard is absolutely terrible.", "complaint"),
    ("What programming language should I learn first?", "question"),
    ("The customer service is useless.", "complaint"),
    ("Can you explain how databases work?", "question"),
    ("I wasted hours on this broken software.", "complaint"),
]

In [ ]:
# Configuration
num_consistency_samples = 5

In [ ]:
# Compare the two approaches

single_correct = 0
consistency_correct = 0
for text, expected in test_data:
assert False, 'Not implemented yet'


### Exercise 3: CoT + Self-Consistency for Reasoning

The most powerful combination: use CoT prompts with self-consistency.
Generate multiple reasoning paths and take the majority answer.

In [ ]:
def self_consistency_cot(
    question: str,
    cot_examples: List[Tuple[str, str, str]],
    num_samples: int = 5,
    temperature: float = 0.7,
) -> Tuple[str, dict]:
    """
    Self-consistency with Chain-of-Thought for reasoning tasks.

    Args:
        question: The question to solve
        cot_examples: CoT examples (question, reasoning, answer)
        num_samples: Number of reasoning paths to generate
        temperature: Temperature for sampling

    Returns:
        Tuple of (majority_answer, vote_counts)
    """
    # Implement CoT + Self-Consistency

    prompt = build_cot_prompt(cot_examples, question)
    full_prompt = build_prompt(prompt)
    responses = generate_text(full_prompt, ..., num_return_sequences=num_samples)
    answers = [extract_answer_from_cot(r) for r in responses]
    assert False, 'Not implemented yet'


### CoT + SELF-CONSISTENCY

In [ ]:
# Test CoT + Self-Consistency

for question, expected in zip(reasoning_questions, expected_answers):
    answer, votes = self_consistency_cot(
        question,
        cot_examples,
        num_samples=num_consistency_samples,
    )
    print("=" * 80)
    print(f"Q: {question}")
    print(f"Votes: {votes}")
    print(f"Answer: {answer} (expected: {expected})")
    correct = "✓" if answer == expected else "✗"
    print(f"Result: {correct}")
    print()

## Part 2: Exploring the LoTTE Dataset

LoTTE (Long-Tail Topic-stratified Evaluation) is an IR benchmark based on
StackExchange Q&A. We use the technology domain with search queries.

Dataset: https://ir-datasets.com/lotte.html

We use PyTerrier's integration with ir-datasets for easy access:
https://ir-datasets.com/pyterrier.html

In [ ]:
# Load the LoTTE technology dataset via PyTerrier's ir-datasets integration
dataset = pt.get_dataset("irds:lotte/technology/dev/search")
print("Dataset loaded: lotte/technology/dev/search")

In [ ]:
# Explore the structure using PyTerrier's dataset methods
print("\nDataset info:")
# Get counts from the underlying ir-datasets object
irds = dataset.irds_ref()
print(f"  - Documents: {irds.docs_count()}")
print(f"  - Queries: {irds.queries_count()}")
print(f"  - Relevance judgments (qrels): {irds.qrels_count()}")

In [ ]:
# Display some documents using PyTerrier's corpus iterator
print("\nExample documents (StackExchange answers):")
for i, doc in enumerate(dataset.get_corpus_iter()):
    if i >= 3:
        break
    print(f"\n[{i+1}] ID: {doc['docno']}")
    print(f"    Text: {doc['text'][:200]}...")

In [ ]:
# Display some queries using PyTerrier's topics method
topics_df = dataset.get_topics()
print("\nExample queries:")
print(topics_df.head())

In [ ]:
# Display some relevance judgments using PyTerrier's qrels method
qrels_df_all = dataset.get_qrels()
print("\nExample qrels (query-document relevance):")
print(qrels_df_all.head())

### Exercise 4: Index the dataset with PyTerrier

Using PyTerrier's integration, we can access the data via:
- `dataset.get_corpus_iter()` for documents
- `dataset.get_topics()` for queries
- `dataset.get_qrels()` for relevance judgments

We'll index the entire corpus with PyTerrier for efficient retrieval.
The document text is stored in the index metadata for later lookup.

In [ ]:
# Get queries and qrels
queries_df = dataset.get_topics()
qrels_df = dataset.get_qrels()

print("Loaded:")
print(f"  - Queries: {len(queries_df)}")
print(f"  - Qrels: {len(qrels_df)}")

In [ ]:
# Index the entire corpus with PyTerrier (or load existing index)
# We store document text in metadata for retrieval during RAG
index_path = Path("./outputs/practical-04/index_lotte").absolute()

# Check if index already exists
if (index_path / "data.properties").exists():
    print(f"Loading existing index from {index_path}")
    index_ref = str(index_path)
else:
    print(f"Creating new index at {index_path}")
    if index_path.is_dir():
        shutil.rmtree(index_path)
    index_path.mkdir(parents=True, exist_ok=True)

    indexer = pt.IterDictIndexer(
        str(index_path),
        overwrite=True,
        meta={"docno": 50, "text": 4096},  # Store text in metadata
        meta_reverse=["docno"],
    )

    # Index the corpus - PyTerrier handles iteration efficiently
    print("Indexing corpus...")
    index_ref = indexer.index(dataset.get_corpus_iter())

# Get index statistics
index = pt.IndexFactory.of(index_ref, memory={"meta": True})
meta_index = index.getMetaIndex()
print(f"Index has {index.getCollectionStatistics().getNumberOfDocuments()} documents")

In [ ]:
# Helper functions to get document text from index (uses cached meta_index)
def get_doc_text(meta_index, doc_id: str) -> str:
    """Retrieve document text from PyTerrier index metadata."""
    try:
        docid = meta_index.getDocument("docno", doc_id)
        if docid >= 0:
            return meta_index.getItem("text", docid)
    except Exception:
        pass
    return ""


def get_text_from_index(meta_index, doc_ids: list[str]) -> dict[str, str]:
    """Retrieve text for multiple documents from the index metadata."""
    result = {}
    for doc_id in doc_ids:
        try:
            docid = meta_index.getDocument("docno", doc_id)
            if docid >= 0:
                result[doc_id] = meta_index.getItem("text", docid)
        except Exception:
            pass
    return result

## Part 3: Generating Training Data for IR

To train a neural retrieval model like SPLADE, we need query-document pairs.
While LoTTE provides some qrels, we can **generate additional queries** using
the LLM to create more training data.

### Why generate synthetic queries?
- More training data improves model quality
- Covers diverse query formulations for the same document
- Cost-effective compared to human annotation

### Strategy:
1. Sample documents from the corpus
2. Use the LLM to generate questions that the document answers
3. Filter generated queries for quality
4. Save for training in the next practical

In [ ]:
# Configuration for query generation
num_docs_to_augment = 50  # Number of documents to generate queries for
queries_per_doc = 2  # Number of queries to generate per document

In [ ]:
def generate_queries_for_document(
    document: str,
    num_queries: int = 2,
    max_doc_length: int = 500,
) -> List[str]:
    """
    Generate search queries that the document would answer.

    Args:
        document: The document text
        num_queries: Number of queries to generate
        max_doc_length: Maximum document length to use

    Returns:
        List of generated queries
    """
    # Implement query generation

    doc_excerpt = document[:max_doc_length]
    prompt = f"Given this document, generate {num_queries} search queries..."
    response = generate_text(build_prompt(prompt), ...)
    assert False, 'Not implemented yet'


In [ ]:
# Test query generation on a sample document
# Get a sample document ID from qrels (guaranteed to exist in index)
sample_doc_id = qrels_df["docno"].iloc[0]
sample_doc = get_doc_text(meta_index, sample_doc_id)

print(f"Document ({sample_doc_id}):")
print(f"{sample_doc[:300]}...")
print()

generated = generate_queries_for_document(sample_doc, num_queries=3)
print("Generated queries:")
for i, q in enumerate(generated, 1):
    print(f"  {i}. {q}")

### Exercise: Generate queries for multiple documents

Generate synthetic training data by creating queries for documents in the corpus.

In [ ]:
def generate_training_pairs(
    index_ref,
    qrels_df: pd.DataFrame,
    num_docs: int = 50,
    queries_per_doc: int = 2,
) -> List[Dict]:
    """
    Generate query-document training pairs.

    Args:
        index_ref: PyTerrier index reference
        qrels_df: DataFrame with qrels (to get document IDs)
        num_docs: Number of documents to process
        queries_per_doc: Queries to generate per document

    Returns:
        List of {"query": str, "doc_id": str, "document": str}
    """
    # Generate training pairs

    # 1. Sample documents from the corpus (use qrels to get doc IDs)
    # 2. Generate queries for each document
    # 3. Create training pairs
    import random
    doc_ids = qrels_df["docno"].unique().tolist()
    sampled_ids = random.sample(doc_ids, min(num_docs, len(doc_ids)))
    training_pairs = []
    for doc_id in tqdm(sampled_ids):
    assert False, 'Not implemented yet'


In [ ]:
# Generate synthetic training pairs
print(f"Generating queries for {num_docs_to_augment} documents...")
synthetic_pairs = generate_training_pairs(
    index_ref,
    qrels_df,
    num_docs=num_docs_to_augment,
    queries_per_doc=queries_per_doc,
)

print(f"\nGenerated {len(synthetic_pairs)} synthetic query-document pairs")

# Show examples
print("\nExample synthetic pairs:")
for pair in synthetic_pairs[:3]:
    print(f"  Query: {pair['query']}")
    print(f"  Doc: {pair['document'][:100]}...")
    print()

### Filtering synthetic queries

Not all generated queries are good. We can filter by:
- Query length (too short or too long queries are often bad)
- Diversity (avoid duplicate queries)

In [ ]:
def filter_synthetic_pairs(
    pairs: List[Dict],
    min_query_words: int = 3,
    max_query_words: int = 20,
) -> List[Dict]:
    """
    Filter synthetic query-document pairs for quality.

    Args:
        pairs: List of training pairs
        min_query_words: Minimum query length
        max_query_words: Maximum query length

    Returns:
        Filtered list of pairs
    """
    filtered = []
    seen_queries = set()

    for pair in pairs:
        query = pair["query"]

        # Length filter
        query_words = len(query.split())
        if query_words < min_query_words or query_words > max_query_words:
            continue

        # Duplicate filter
        query_lower = query.lower()
        if query_lower in seen_queries:
            continue
        seen_queries.add(query_lower)

        filtered.append(pair)

    return filtered

In [ ]:
# Filter the synthetic pairs
filtered_pairs = filter_synthetic_pairs(synthetic_pairs)

print(f"Filtered: {len(synthetic_pairs)} -> {len(filtered_pairs)} pairs")
print(f"Kept: {100 * len(filtered_pairs) / max(1, len(synthetic_pairs)):.1f}%")

### Part 3b: Validating Generated Queries with Evaluation Metrics

How do we know if our generated queries are good? We can compare them to
**real user queries** from the LoTTE dataset using evaluation metrics.

### Evaluation Metrics

- **ROUGE**: Measures n-gram overlap between texts (recall-oriented)
- **BERTScore**: Measures semantic similarity using BERT embeddings

### Validation strategy:
1. Select documents that have real queries (from qrels)
2. Generate multiple synthetic queries for each document
3. Compare generated queries to the real query for that document
4. Take the **max score** over generations (best of N)

This approach directly compares against ground truth without needing
embedding models to find similar queries.

In [ ]:
# Initialize evaluation metrics
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
bertscore = load_metric("bertscore")

In [ ]:
def compute_rouge(prediction: str, reference: str) -> dict:
    """Compute ROUGE scores between two texts."""
    scores = rouge.score(reference, prediction)
    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def compute_bertscore(predictions: List[str], references: List[str]) -> dict:
    """Compute BERTScore for a batch of predictions."""
    results = bertscore.compute(
        predictions=predictions,
        references=references,
        lang="en",
        model_type="microsoft/deberta-xlarge-mnli",
    )
    return {
        "precision": sum(results["precision"]) / len(results["precision"]),
        "recall": sum(results["recall"]) / len(results["recall"]),
        "f1": sum(results["f1"]) / len(results["f1"]),
    }

In [ ]:
def validate_query_generation(
    index_ref,
    queries_df: pd.DataFrame,
    qrels_df: pd.DataFrame,
    num_samples: int = 10,
    queries_per_doc: int = 3,
) -> Dict:
    """
    Validate query generation by comparing to real queries for the same document.

    For documents with known queries, generate synthetic queries and compare
    to the ground truth. Takes the max score over multiple generations.

    Args:
        index_ref: PyTerrier index reference
        queries_df: DataFrame with real queries
        qrels_df: DataFrame with relevance judgments
        num_samples: Number of documents to evaluate
        queries_per_doc: Number of queries to generate per document

    Returns:
        Dictionary with validation metrics and examples
    """
    # Implement query validation

    doc_to_queries = {}  # Map doc_id -> list of real queries
    for _, row in qrels_df.iterrows():
    assert False, 'Not implemented yet'


In [ ]:
# Validate query generation quality
print("Evaluating query generation quality...")
print("(Generating queries for documents with known real queries)\n")

num_validation_samples = 10


validation_results = validate_query_generation(
    index_ref,
    queries_df,
    qrels_df,
    num_samples=num_validation_samples,
    queries_per_doc=3,
)

if "error" not in validation_results:
    print("=== Query Generation Validation ===")
    print(f"Max ROUGE-1 (avg): {validation_results['avg_max_rouge1']:.3f}")
    print(f"Max ROUGE-L (avg): {validation_results['avg_max_rougeL']:.3f}")
    print(f"BERTScore (best): {validation_results['avg_bertscore']:.3f}")

    # Show examples
    examples = validation_results["examples"]
    examples_sorted = sorted(examples, key=lambda x: x["max_rouge1"], reverse=True)

    print("\n--- Best Examples (highest max ROUGE-1) ---")
    for ex in examples_sorted[:2]:
        print(f"  Real query: {ex['real_query']}")
        print(f"  Best generated: {ex['best_generated']}")
        print(f"  Max ROUGE-1: {ex['max_rouge1']:.3f}")
        print(f"  All generated: {ex['generated_queries']}")
        print()

    print("--- Challenging Examples (lower ROUGE) ---")
    for ex in examples_sorted[-2:]:
        print(f"  Real query: {ex['real_query']}")
        print(f"  Best generated: {ex['best_generated']}")
        print(f"  Max ROUGE-1: {ex['max_rouge1']:.3f}")
        print()
else:
    print(f"Validation skipped: {validation_results['error']}")

### Interpretation of Validation Results

We use **max over generations** because:
- Query generation is inherently variable
- Any of the generated queries could be good
- This measures "can the model generate at least one good query?"

**Metrics interpretation**:
- **Max ROUGE-1/L**: N-gram overlap with real query (best of N generations)
- **BERTScore**: Semantic similarity of best generation to real query

**Quality indicators**:
- Max ROUGE-1 > 0.4: Good lexical overlap
- Max ROUGE-1 > 0.6: Excellent - very similar phrasing
- BERTScore > 0.7: Strong semantic similarity

Low ROUGE but high BERTScore means the generated query captures
the intent but uses different words (which is fine for diversity).

### Combining real and synthetic data

We combine the original qrels with our synthetic pairs for training.

In [ ]:
def create_combined_training_data(
    index_ref,
    qrels_df: pd.DataFrame,
    queries_df: pd.DataFrame,
    synthetic_pairs: List[Dict],
) -> List[Dict]:
    """
    Combine real qrels with synthetic pairs.

    Args:
        index_ref: PyTerrier index reference
        qrels_df: Original relevance judgments
        queries_df: Original queries
        synthetic_pairs: Synthetically generated pairs

    Returns:
        Combined list of training pairs
    """
    training_data = []

    # Add real pairs from qrels
    for _, row in qrels_df.iterrows():
        qid = row["qid"]
        doc_id = row["docno"]

        query_rows = queries_df[queries_df["qid"] == qid]["query"].values
        if len(query_rows) == 0:
            continue

        doc_text = get_doc_text(meta_index, doc_id)
        if not doc_text:
            continue

        training_data.append(
            {
                "query": query_rows[0],
                "doc_id": doc_id,
                "document": doc_text,
                "source": "qrels",  # From original dataset
            }
        )

    # Add synthetic pairs
    training_data.extend(synthetic_pairs)

    return training_data

In [ ]:
# Create combined training data
combined_training = create_combined_training_data(
    index_ref, qrels_df, queries_df, filtered_pairs
)

print(f"Combined training data: {len(combined_training)} pairs")
print(f"  - From qrels: {sum(1 for p in combined_training if p['source'] == 'qrels')}")
print(
    f"  - Synthetic: {sum(1 for p in combined_training if p['source'] == 'synthetic')}"
)

### Save training data for practical 5

Save the combined training data for SPLADE training.

In [ ]:
import json

# Prepare data for saving (don't include full document text to save space)
training_export = []
for pair in combined_training:
    training_export.append(
        {
            "query": pair["query"],
            "doc_id": pair["doc_id"],
            "source": pair["source"],
        }
    )

output_dir = Path("./outputs/practical-04")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "training_data_for_splade.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(training_export, f, indent=2)

print(f"Saved {len(training_export)} training pairs to {output_path}")

## Part 4: Introduction to RAG with BM25

Before implementing SPLADE in the next practical, let's see how RAG
(Retrieval-Augmented Generation) works with a simple BM25 retriever.

RAG combines:
1. **Retrieval**: Find relevant documents for a question
2. **Generation**: Use the documents as context to generate an answer

We already indexed the documents in Part 2, so we just need to create
the BM25 retriever.

In [ ]:
# Create BM25 retriever using the index we created earlier
bm25 = pt.BatchRetrieve(index_ref, wmodel="BM25")
print("BM25 retriever ready")

In [ ]:
def simple_rag(
    question: str,
    retriever,
    index_ref,
    model,
    tokenizer,
    top_k: int = 3,
) -> dict:
    """
    Simple RAG pipeline with BM25.

    Args:
        question: User question
        retriever: PyTerrier retriever (BM25)
        index_ref: PyTerrier index reference (for text retrieval)
        model: Generation model
        tokenizer: Model tokenizer
        top_k: Number of documents to retrieve

    Returns:
        Dict with retrieved docs and generated answer
    """
    # Implement simple RAG with BM25

    results = retriever.search(question)
    top_docs = results.head(top_k)
    doc_ids = top_docs["docno"].tolist()
    doc_texts = get_text_from_index(index_ref, doc_ids)
    context_parts = [doc_texts.get(doc_id, "")[:300] for doc_id in doc_ids]
    assert False, 'Not implemented yet'


In [ ]:
# Test RAG with BM25
test_questions_rag = [
    "How do I install Python packages?",
    "What is the difference between a list and a tuple?",
]

print("=== RAG with BM25 ===\n")
for question in test_questions_rag[:1]:  # Just one for the practical
    result = simple_rag(question, bm25, index_ref, model, tokenizer, top_k=3)

    print(f"Question: {result['question']}")
    print(f"\nRetrieved documents ({len(result['retrieved_docs'])}):")
    for doc_info in result["retrieved_docs"]:
        print(f"  - {doc_info['doc_id'][:20]}... (score: {doc_info['score']:.2f})")
    print(f"\nGenerated answer:\n{result['answer'][:500]}")
    print("\n" + "=" * 80)

### Exercise 5: Compare retrieval without vs with generation

Let's see if retrieval helps the model answer questions better.

In [ ]:
def answer_without_retrieval(question: str, model, tokenizer) -> str:
    """Generate an answer without retrieval (direct generation)."""
    messages = [
        {"role": "system", "content": "You are a helpful technical assistant."},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in answer.lower():
        answer = answer.split("assistant")[-1].strip()
    return answer

In [ ]:
# Compare approaches
comparison_question = "How do I debug a Python program?"

print(f"Question: {comparison_question}\n")

# Without retrieval
print("=== WITHOUT RETRIEVAL ===")
direct_answer = answer_without_retrieval(comparison_question, model, tokenizer)
print(direct_answer[:400])

print("\n" + "=" * 80)

# With RAG
print("\n=== WITH RAG (BM25) ===")
rag_result = simple_rag(comparison_question, bm25, index_ref, model, tokenizer)
print(rag_result["answer"][:400])

### Discussion: When is RAG useful?

RAG is particularly useful when:
- The knowledge is specific or recent (not in the model's training data)
- We need citations or sources for the answers
- The corpus is specialized (like StackExchange technical Q&A)

In the next practical, we will replace BM25 with SPLADE for better retrieval.